# HybridCC VOD — Colab notebook

End-to-end test of the VOD CEA-608 pipeline.

```
input.mp4 ─► faster-whisper ─► input.vtt
                                  │
input.mp4 ──┬─────────────────────┘
            ▼
         ffmpeg (mp4 → flv pipe)
            │
            ▼
         hybridCC-vod  ◄── input.vtt   (PTS-matched CEA-608 SEI)
            │
            ▼
         ffmpeg (flv → mp4, -a53cc 1)
            │
            ▼
         output.mp4 with embedded CEA-608
```

**Runtime:** GPU (free-tier T4 is enough). Run cells top-to-bottom.

## 1 · GPU + disk check

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU — Runtime → Change runtime type → GPU'
!df -h / | tail -1

Tesla T4, 15360 MiB
overlay         113G   44G   70G  39% /


## 2 · Install build deps + ffmpeg

Colab images already have build-essential + cmake. ffmpeg is preinstalled. re2c is optional but available.

In [2]:
!apt-get -qq install -y build-essential cmake git ffmpeg re2c 2>&1 | tail -3
!gcc --version | head -1
!cmake --version | head -1
!ffmpeg -version 2>&1 | head -1

Unpacking re2c (3.0-1) ...
Setting up re2c (3.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
cmake version 3.31.10
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


## 3 · Build libcaption (upstream, MIT)

In [3]:
%cd /content
![ -d libcaption ] || git clone --depth 1 https://github.com/szatmary/libcaption.git
%cd /content/libcaption
# Verbose — show actual cmake/make output so we can see failures
!cmake . -DENABLE_RE2C=ON 2>&1 | tail -15
!echo '--- make ---'
!make -j$(nproc) 2>&1 | tail -15
!echo '\n--- build artifacts (.a / .so) ---'
!find . -maxdepth 3 \( -name '*.a' -o -name '*.so*' \) 2>/dev/null
!echo '\n--- src/ .c files (used by fallback compile if no lib found) ---'
!ls src/*.c 2>/dev/null | head -20

/content
Cloning into 'libcaption'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 69 (delta 20), reused 12 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 164.48 KiB | 2.32 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/libcaption
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found re2c: /usr/bin/re2c
-- Could NOT find Doxygen (missing: DOXYGEN_EXECUTABLE) 
-- Configuring done (0.8s)
-- Generating done (0.0s)
-- Build files have been written to: /content/libcaption
--- make ---
[ 81%] Linking 

## 4 · Write `hybridCC-vod.c`

Embedded source for v1. Once `hybridcc/` is on GitHub, replace this cell with `git clone`.

In [4]:
from pathlib import Path

VOD_C = r'''
/*
 * hybridCC-vod -- CEA-608 caption injector for VOD (timestamp-matched)
 * Reads FLV from stdin, parses VTT cues, injects CEA-608 SEI by PTS.
 * Writes captioned FLV to stdout.
 */

#include "caption/caption.h"
#include "flv.h"
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#ifdef _WIN32
#include <io.h>
#include <fcntl.h>
#endif

#define MAX_VTT_SIZE  (4 * 1024 * 1024)
#define MAX_CUES       20000
#define MAX_CUE_TEXT   1024

typedef struct {
    double start;
    double end;
    char   text[MAX_CUE_TEXT];
} vtt_cue_t;

static vtt_cue_t g_cues[MAX_CUES];
static int g_cue_count = 0;
static int g_cursor    = 0;

static double parse_ts(const char* s, int len)
{
    char buf[32];
    if (len <= 0 || len >= (int)sizeof(buf)) return -1.0;
    memcpy(buf, s, len);
    buf[len] = 0;
    int h = 0, m = 0, sec = 0, ms = 0;
    if (sscanf(buf, "%d:%d:%d.%d", &h, &m, &sec, &ms) == 4)
        return h * 3600.0 + m * 60.0 + sec + ms / 1000.0;
    if (sscanf(buf, "%d:%d.%d", &m, &sec, &ms) == 3)
        return m * 60.0 + sec + ms / 1000.0;
    return -1.0;
}

static int load_vtt_cues(const char* path)
{
    FILE* f = fopen(path, "rb");
    if (!f) { fprintf(stderr, "[CC] Cannot open VTT: %s\n", path); return 0; }
    fseek(f, 0, SEEK_END);
    long sz = ftell(f);
    fseek(f, 0, SEEK_SET);
    if (sz <= 0 || sz >= MAX_VTT_SIZE) { fclose(f); return 0; }
    char* buf = (char*)malloc((size_t)sz + 1);
    if (!buf) { fclose(f); return 0; }
    if (fread(buf, 1, (size_t)sz, f) != (size_t)sz) { free(buf); fclose(f); return 0; }
    buf[sz] = 0;
    fclose(f);

    char* p = buf;
    while (*p && g_cue_count < MAX_CUES) {
        char* arrow = strstr(p, "-->");
        if (!arrow) break;
        char* line_start = arrow;
        while (line_start > buf && line_start[-1] != '\n') line_start--;
        char* s_end = arrow;
        while (s_end > line_start && (s_end[-1] == ' ' || s_end[-1] == '\t')) s_end--;
        double start = parse_ts(line_start, (int)(s_end - line_start));
        char* e_start = arrow + 3;
        while (*e_start == ' ' || *e_start == '\t') e_start++;
        char* e_end = e_start;
        while (*e_end && *e_end != ' ' && *e_end != '\r' && *e_end != '\n') e_end++;
        double end = parse_ts(e_start, (int)(e_end - e_start));
        if (start < 0 || end <= start) { p = arrow + 3; continue; }
        char* text_start = strchr(arrow, '\n');
        if (!text_start) break;
        text_start++;
        char* text_end = strstr(text_start, "\r\n\r\n");
        if (!text_end) text_end = strstr(text_start, "\n\n");
        if (!text_end) text_end = buf + sz;
        vtt_cue_t* cue = &g_cues[g_cue_count];
        int outlen = 0, prev_space = 1;
        int span = (int)(text_end - text_start);
        for (int i = 0; i < span; i++) {
            char c = text_start[i];
            if (c == '\r' || c == '\n' || c == '\t') c = ' ';
            if (c == ' ' && prev_space) continue;
            if (outlen >= MAX_CUE_TEXT - 1) break;
            cue->text[outlen++] = c;
            prev_space = (c == ' ');
        }
        while (outlen > 0 && cue->text[outlen-1] == ' ') outlen--;
        cue->text[outlen] = 0;
        if (outlen > 0) {
            cue->start = start;
            cue->end   = end;
            g_cue_count++;
        }
        p = text_end;
    }
    free(buf);
    fprintf(stderr, "[CC] Parsed %d cues from %s\n", g_cue_count, path);
    return g_cue_count;
}

static const char* find_cue_at(double pts_sec)
{
    while (g_cursor < g_cue_count && g_cues[g_cursor].end <= pts_sec) g_cursor++;
    if (g_cursor >= g_cue_count) return NULL;
    if (pts_sec >= g_cues[g_cursor].start) return g_cues[g_cursor].text;
    return NULL;
}

int main(int argc, char** argv)
{
    if (argc < 2) {
        fprintf(stderr, "Usage: %s captions.vtt < input.flv > output.flv\n", argv[0]);
        return 1;
    }
#ifdef _WIN32
    _setmode(_fileno(stdin), _O_BINARY);
    _setmode(_fileno(stdout), _O_BINARY);
#endif
    load_vtt_cues(argv[1]);
    flvtag_t tag;
    flvtag_init(&tag);
    int has_audio = 0, has_video = 0;
    if (!flv_read_header(stdin, &has_audio, &has_video)) {
        fprintf(stderr, "[CC] Not a valid FLV on stdin\n");
        return 1;
    }
    flv_write_header(stdout, has_audio, has_video);
    fprintf(stderr, "[CC] Streaming (audio=%d video=%d)...\n", has_audio, has_video);
    long video_frames = 0, injected = 0;
    while (flv_read_tag(stdin, &tag)) {
        if (flvtag_avcpackettype_nalu == flvtag_avcpackettype(&tag)) {
            video_frames++;
            uint32_t pts_ms = flvtag_pts(&tag);
            const char* text = find_cue_at(pts_ms / 1000.0);
            if (text && *text) {
                flvtag_addcaption_text(&tag, (const utf8_char_t*)text);
                injected++;
            }
        }
        flv_write_tag(stdout, &tag);
    }
    fprintf(stderr, "[CC] Done: %ld frames, %ld with captions\n", video_frames, injected);
    flvtag_free(&tag);
    return 0;
}
'''

Path('/content/hybridCC-vod.c').write_text(VOD_C)
print('wrote /content/hybridCC-vod.c -', len(VOD_C), 'chars')

wrote /content/hybridCC-vod.c - 5010 chars


In [9]:
import subprocess, glob, os, sys

os.chdir('/content')

# Strategy: try linking against the cmake-built lib first.
# If no lib found, compile all libcaption sources directly with hybridCC-vod.c.
libs = glob.glob('libcaption/libcaption.a') + glob.glob('libcaption/build/libcaption.a') \
     + glob.glob('libcaption/libcaption.so*') + glob.glob('libcaption/build/libcaption.so*')

base = ['gcc', '-O2', '-Wall', '-I', 'libcaption', '-I', 'libcaption/src', '-I', 'libcaption/examples', '-I', 'libcaption/caption',
        '-o', 'hybridCC-vod', 'hybridCC-vod.c']

if libs:
    print(f'[build] linking against {libs[0]}')
    # Include flv.c from examples/ explicitly, as its functions are used by hybridCC-vod.c
    # and it's not part of libcaption.a which comes from src/.
    cmd = base + ['libcaption/examples/flv.c'] + [libs[0], '-lm']
else:
    src_files = sorted(glob.glob('libcaption/src/*.c'))
    if not src_files:
        sys.exit('[build] FAIL: no libcaption.a AND no libcaption/src/*.c — clone may have failed')
    print(f'[build] no static lib found — compiling {len(src_files)} libcaption sources directly')
    # If no static lib, compile all src/*.c and flv.c directly
    cmd = base + src_files + ['libcaption/examples/flv.c'] + ['-lm']

print('[build] cmd:', ' '.join(cmd[:8]), '...')
r = subprocess.run(cmd, capture_output=True, text=True)
if r.stdout: print(r.stdout)
if r.stderr: print(r.stderr)
if r.returncode != 0:
    sys.exit(f'[build] FAIL exit={r.returncode}')

print('\n[build] OK')
subprocess.run(['ls', '-la', '/content/hybridCC-vod'])
print('\n[build] usage smoke test:')
subprocess.run(['/content/hybridCC-vod'], stderr=subprocess.STDOUT)

[build] linking against libcaption/libcaption.a
[build] cmd: gcc -O2 -Wall -I libcaption -I libcaption/src -I ...

[build] OK

[build] usage smoke test:


CompletedProcess(args=['/content/hybridCC-vod'], returncode=1)

## 5 · Upload a test MP4

Short clip (30s–2min) with clear English speech for the first run.

In [10]:
from google.colab import files
import shutil, os
uploaded = files.upload()
src_name = list(uploaded.keys())[0]
shutil.move(src_name, '/content/input.mp4')
print('input.mp4 ready -', os.path.getsize('/content/input.mp4'), 'bytes')
!ffprobe -v error -show_entries stream=codec_name,codec_type -of default=nw=1 /content/input.mp4

Saving greenscreenOpusClip.mp4 to greenscreenOpusClip.mp4
input.mp4 ready - 2367038 bytes
codec_name=h264
codec_type=video
codec_name=aac
codec_type=audio


## 6 · Whisper transcribe → VTT

`large-v3` for quality, `medium.en` for speed.

In [11]:
!pip -q install faster-whisper==1.0.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.7/34.7 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 74.8 MB/s eta 0:00:00


In [12]:
import time
from faster_whisper import WhisperModel

MODEL = 'large-v3'
DEVICE = 'cuda'
COMPUTE = 'float16'

t0 = time.time()
model = WhisperModel(MODEL, device=DEVICE, compute_type=COMPUTE)
print(f'model loaded in {time.time()-t0:.1f}s')

t0 = time.time()
segments, info = model.transcribe('/content/input.mp4', beam_size=5, language='en')
segments = list(segments)
elapsed = time.time() - t0
print(f'transcribed {info.duration:.1f}s of audio in {elapsed:.1f}s ({info.duration/elapsed:.1f}x realtime)')

def fmt_ts(s):
    h = int(s // 3600)
    m = int((s % 3600) // 60)
    sec = s % 60
    return f'{h:02d}:{m:02d}:{sec:06.3f}'

with open('/content/input.vtt', 'w') as f:
    f.write('WEBVTT\n\n')
    for s in segments:
        text = (s.text or '').strip()
        if not text:
            continue
        f.write(f'{fmt_ts(s.start)} --> {fmt_ts(s.end)}\n{text}\n\n')

print(f'\n{len(segments)} cues written to /content/input.vtt')
!head -20 /content/input.vtt

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model loaded in 32.4s
transcribed 16.0s of audio in 1.9s (8.4x realtime)

1 cues written to /content/input.vtt
WEBVTT

00:00:00.000 --> 00:00:16.020
even now i miss that stubborn goat now go before i start crying into my mead the tale is done but her spirit lives on every time the wind howls from the north



## 7 · Inject CEA-608 via the FLV pipe

In [13]:
%cd /content
!set -o pipefail; \
  ffmpeg -y -hide_banner -loglevel warning \
    -i input.mp4 -c:v copy -c:a aac -ac 2 -ar 44100 -f flv pipe:1 \
  | ./hybridCC-vod input.vtt \
  | ffmpeg -y -hide_banner -loglevel warning \
    -f flv -i pipe:0 -c:v copy -c:a copy -a53cc 1 -movflags +faststart output.mp4 \
  && ls -la output.mp4 || echo 'pipeline error'

/content/libcaption
[CC] Parsed 1 cues from input.vtt
[CC] Streaming (audio=4 video=1)...
[flv @ 0x5a0eb0a9b5c0] Failed to update header with correct duration.
[flv @ 0x5a0eb0a9b5c0] Failed to update header with correct filesize.
[CC] Done: 385 frames, 384 with captions
-rw-r--r-- 1 root root 2268471 May  7 01:57 output.mp4


## 8 · Verify CEA-608 landed

ffprobe sets `closed_captions=1` on the video stream when CEA-608 SEI is detected. We also try to round-trip extract via `[out0+subcc]`.

In [14]:
print('=== ffprobe stream summary ===')
!ffprobe -v error -show_streams -select_streams v:0 /content/output.mp4 | grep -E 'codec_name|width|height|closed_captions|duration='
print('\n(look for closed_captions=1)\n')

print('=== Extract CEA-608 from output (first 30s) ===')
!ffmpeg -hide_banner -loglevel error -t 30 -f lavfi -i "movie=/content/output.mp4[out0+subcc]" -map 0:s -c:s webvtt -y /content/extracted.vtt 2>&1 || echo 'extraction failed (try a longer clip)'
!head -30 /content/extracted.vtt 2>/dev/null || echo 'no extracted.vtt produced'

=== ffprobe stream summary ===
codec_name=h264
width=720
height=406
coded_width=720
coded_height=406
closed_captions=0
duration=16.041563

(look for closed_captions=1)

=== Extract CEA-608 from output (first 30s) ===
WEBVTT

00:00.000 --> 00:00.007
even now i miss that stubborn
goat now go before i start
crying into my mead the tale is
done but her spirit lives on
every time the wind howls from
the north

00:00.007 --> 00:00.014
even now i miss that stubborn
goat now go before i start
crying into my mead the tale is
done but her spirit lives on
every time the wind howls from
the north

00:00.015 --> 00:00.022
even now i miss that stubborn
goat now go before i start
crying into my mead the tale is
done but her spirit lives on
every time the wind howls from
the north

00:00.022 --> 00:00.029
even now i miss that stubborn
goat now go before i start
crying into my mead the tale is


## 9 · Download captioned MP4 + source VTT

In [15]:
from google.colab import files
files.download('/content/output.mp4')
files.download('/content/input.vtt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next steps

- **Modal port** — same image add-ons, same compile, drop into `modal-caption-server.py` as an `InjectWorker` class with a new `POST /caption/inject` endpoint.
- **Hetzner live add-on** — `src/legacy/hybridCC-stdin.c` builds the same way; same notebook minus Whisper, plug into MediaMTX.
- **Windows build** — MSVC 2019 + libcaption's CMakeLists. Same source, the `#ifdef _WIN32` blocks reactivate the `_setmode` patch.
- **Format options** — replace `-a53cc 1` with `-c:s mov_text` for a web-friendly VTT track instead of (or alongside) CEA-608 SEI.